## Merging All Building Data Files

The following notebook takes 8 parquet files downloaded off of [Google Earth Engine](https://sat-io.earthengine.app/view/gba) and merges them into one file. 

The purpose of this is to contain a singular file with building footprint and height information, for only California.

The method of joining all 8 files will simply be to:
1. Load in all 7 parquet files that correspond to California
2. Concatenate them together

The file is then saved as a GeoJSON, which will be read in during the remainder of our analysis.

In [1]:
import os
import geopandas as gpd
import pandas as pd

Parquet files are labeled as follows, corresponding to tiles labeled in the image below:

<div>
<img src="images/labeled_tiles.png" width="500"/>
</div>

- **1**: w125_n45_w120_n40.parquet (leftmost/contains Santa Barbara 
- **2**: w120_n35_w115_n30 (contains LA and San Diego)
- **3**: w115_n35_w110_n30 (contains Arizona/California border)
- **4**: w125_n40_w120_n35 (contains San Francisco and Sacramento) 
- **5**: w120_n40_w115_n35 (contains Nevada/California border)
- **6**: w115_n40_w110_n35 (contains bit of California that is below Las Vegas)
- **7**: w125_n45_w120_n40 (topmost!)

They will be joined in numerical order (1-7).

In [2]:
# Load all parquets (takes about 2 min)
fp = os.path.join('data', 'building_parquets', 'w125_n45_w120_n40.parquet')
tile_1 = gpd.read_parquet(fp).to_crs(epsg=4326)

fp = os.path.join('data', 'building_parquets', 'w120_n35_w115_n30.parquet')
tile_2 = gpd.read_parquet(fp).to_crs(epsg=4326)

fp = os.path.join('data', 'building_parquets', 'w115_n35_w110_n30.parquet')
tile_3 = gpd.read_parquet(fp).to_crs(epsg=4326)

fp = os.path.join('data', 'building_parquets', 'w125_n40_w120_n35.parquet')
tile_4 = gpd.read_parquet(fp).to_crs(epsg=4326)

fp = os.path.join('data', 'building_parquets', 'w120_n40_w115_n35.parquet')
tile_5 = gpd.read_parquet(fp).to_crs(epsg=4326)

fp = os.path.join('data', 'building_parquets', 'w115_n40_w110_n35.parquet')
tile_6 = gpd.read_parquet(fp).to_crs(epsg=4326)

fp = os.path.join('data', 'building_parquets', 'w125_n45_w120_n40.parquet')
tile_7 = gpd.read_parquet(fp).to_crs(epsg=4326)

In [3]:
buildings_ca = pd.concat([tile_1, tile_2, tile_3, tile_4, tile_5, tile_6, tile_7])

In [4]:
# make sure the new data frame contains all of the necessary rows
assert {tile_1.shape[0] + tile_2.shape[0] + tile_3.shape[0] + tile_4.shape[0] + 
        tile_5.shape[0] + tile_6.shape[0] + tile_7.shape[0] == buildings_ca.shape[0]}

In [5]:
# take a quick look
buildings_ca.head()

,source,id,height,var,region,bbox,geometry
0,ms,UnitedStates_023001113_5829,3.622120,1.353466,USA,"{'xmin': -123.78071811077251, 'ymin': 39.99991...","POLYGON ((-123.78072 40.00002, -123.78072 39.9..."
1,ms,UnitedStates_023010002_3040,2.946269,1.295380,USA,"{'xmin': -123.7324915995028, 'ymin': 39.999764...","POLYGON ((-123.73228 39.99976, -123.73220 39.9..."
2,ms,UnitedStates_023010002_982,4.060072,1.137501,USA,"{'xmin': -123.63369298664888, 'ymin': 39.99986...","POLYGON ((-123.63369 39.99991, -123.63363 39.9..."
3,ms,UnitedStates_023010002_1378,8.423013,1.055027,USA,"{'xmin': -123.54532288183235, 'ymin': 39.99993...","POLYGON ((-123.54532 40.00008, -123.54520 39.9..."
4,ms,UnitedStates_023010002_2806,-999.000000,-999.000000,USA,"{'xmin': -123.2800922209761, 'ymin': 39.999886...","POLYGON ((-123.27987 39.99989, -123.27984 39.9..."


In [6]:
os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [8]:
# save as parquet file (quickest way)
buildings_ca.to_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet")

In [ ]:
# # save geodataframe
# buildings_ca.to_file("../../../../../capstone/electrigrid/data/buildings/buildings_ca.geojson", driver='GeoJSON')